In [11]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from databricks.sdk.runtime import dbutils

from keyword_extractor import add_education_level_column, add_experience_years_column, add_skills_column

In [12]:
spark = DatabricksSession.builder.getOrCreate()

In [ ]:
TABLE_NAME = "workspace.silver.job_postings"
VOLUME_PATH = "dbfs:/Volumes/workspace/bronze/raw_data/"

raw_df = spark.read.json(VOLUME_PATH, multiLine=True)

In [23]:
raw_df.show()

+--------------------+--------------------+--------------------+------+
|                data|          parameters|          request_id|status|
+--------------------+--------------------+--------------------+------+
|{EuIDCqIDQUppVDR0...|{us, en, 1, data ...|0d635461-91b3-472...|    OK|
|{EuIDCqIDQUppVDR0...|{us, en, 1, data ...|a0ba55cc-fa93-470...|    OK|
+--------------------+--------------------+--------------------+------+



In [14]:
# explode nested data
jobs_df = raw_df.select(F.explode('data.jobs').alias('job'))

In [15]:
# flatten and select needed fields 
silver_df = jobs_df.select(
    F.col("job.job_id").alias("job_id"),
    F.col("job.job_title").alias("job_title"),
    F.col("job.employer_name").alias("employer_name"),
    F.col("job.employer_logo").alias("employer_logo"),
    F.col("job.job_employment_type").alias("employment_type"),
    F.col("job.job_is_remote").cast("boolean").alias("is_remote"),
    F.col("job.job_city").alias("city"),
    F.col("job.job_state").alias("state"),
    F.col("job.job_country").alias("country"),
    F.col("job.job_min_salary").cast("double").alias("min_salary"),
    F.col("job.job_max_salary").cast("double").alias("max_salary"),
    F.col("job.job_salary_period").alias("salary_period"),
    F.col("job.job_description").alias("job_description"),
    F.from_unixtime(F.col("job.job_posted_at_timestamp")).cast("timestamp").alias("posted_at"),
    F.col("job.job_publisher").alias("publisher"),
    F.col("job.job_apply_link").alias("apply_link"),
)

In [16]:
# drop duplicate jobs
silver_df = silver_df.dropDuplicates(["job_id"])

In [17]:
# remove rows with no job_id
silver_df = silver_df.filter(F.col('job_id').isNotNull()).withColumn('_ingested_at', F.current_timestamp())

In [18]:
# add skill keyword column
silver_df = add_skills_column(silver_df, text_col="job_description")

In [19]:
# add experience years
silver_df = add_experience_years_column(silver_df, text_col="job_description")

In [20]:
# add education level
silver_df = add_education_level_column(silver_df, text_col="job_description")

In [22]:
silver_df.show(100)

+--------------------+--------------------+--------------------+--------------------+---------------+---------+--------------+--------------------+-------+----------+----------+-------------+--------------------+-------------------+------------------+--------------------+--------------------+--------------------+------------------------------+-------------------------+
|              job_id|           job_title|       employer_name|       employer_logo|employment_type|is_remote|          city|               state|country|min_salary|max_salary|salary_period|     job_description|          posted_at|         publisher|          apply_link|        _ingested_at|              skills|extracted_min_years_experience|extracted_education_level|
+--------------------+--------------------+--------------------+--------------------+---------------+---------+--------------+--------------------+-------+----------+----------+-------------+--------------------+-------------------+------------------+-----

In [23]:
silver_df.count()

30

In [24]:
# merge new data logic
if spark.catalog.tableExists(TABLE_NAME):
    target = DeltaTable.forName(spark, TABLE_NAME)
    (target.alias('t')
        .merge(silver_df.alias('s'), "t.job_id = s.job_id") 
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    silver_df.write.format('delta').saveAsTable('workspace.silver.job_postings')

In [25]:
# pull insert/update counts from merge metrics
metrics = spark.sql(f"DESCRIBE HISTORY {TABLE_NAME} LIMIT 1").collect()[0]['operationMetrics']
rows_changed = int(metrics.get('numTargetRowsInserted', 0)) + int(metrics.get('numTargetRowsUpdated', 0))

dbutils.jobs.taskValues.set(key="silver_rows_changed", value="true" if rows_changed > 0 else "false")

In [26]:
metrics

{'numTargetRowsCopied': '0',
 'numTargetRowsDeleted': '0',
 'numTargetBytesRemoved': '45806',
 'numTargetDeletionVectorsAdded': '0',
 'numTargetRowsMatchedUpdated': '20',
 'numTargetRowsMatchedDeleted': '0',
 'numTargetRowsUpdated': '20',
 'numTargetChangeFilesAdded': '0',
 'numTargetRowsNotMatchedBySourceDeleted': '0',
 'rewriteTimeMs': '3620',
 'numTargetFilesAdded': '1',
 'numTargetBytesAdded': '68268',
 'executionTimeMs': '10499',
 'materializeSourceTimeMs': '1820',
 'numTargetRowsInserted': '10',
 'numTargetDeletionVectorsUpdated': '0',
 'scanTimeMs': '4757',
 'numOutputRows': '30',
 'numTargetDeletionVectorsRemoved': '0',
 'numTargetRowsNotMatchedBySourceUpdated': '0',
 'numSourceRows': '30',
 'numTargetFilesRemoved': '1'}